In [23]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder \
    .appName("Tugas7") \
    .master("local[*]") \
    .getOrCreate() 
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [24]:
!hdfs dfs -ls /user/irkham/tugas7/raw/

Found 3 items
-rw-r--r--   1 irkham supergroup        321 2026-09-30 20:55 /user/irkham/tugas7/raw/tugas7_cabang.json
-rw-r--r--   1 irkham supergroup        432 2026-09-30 20:55 /user/irkham/tugas7/raw/tugas7_menu.csv
-rw-r--r--   1 irkham supergroup      52539 2026-09-30 20:55 /user/irkham/tugas7/raw/tugas7_transaksi.csv


In [33]:
# A. EXTRACT

# Membaca data transaksi dari HDFS
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/irkham/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)

# Membaca data cabang dari HDFS
df_cabang = spark.read.json("hdfs://localhost:9000/user/irkham/tugas7/raw/tugas7_cabang.json")

# Membaca data menu dari HDFS
df_menu = spark.read.csv("hdfs://localhost:9000/user/irkham/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

print("Jumlah transaksi :", df_transaksi.count())
print("Jumlah cabang    :", df_cabang.count())
print("Jumlah menu      :", df_menu.count())

Jumlah transaksi : 4000
Jumlah cabang    : 3
Jumlah menu      : 15


In [35]:
# B. TRANSFORM — Join & Kolom Turunan 
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner").join(df_menu, on="menu_id", how="inner")

df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))

df_gabungan.show(truncate=False)

+-------+---------+------+---+--------------+----------+-------------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|kota      |nama_cabang              |nama_menu|kategori_menu |harga|total_penjualan|
+-------+---------+------+---+--------------+----------+-------------------------+---------+--------------+-----+---------------+
|14     |1        |KN0   |4  |Nadia         |Magelang  |Kopi Nusantara Magelang  |Menu-14  |Kopi          |18000|72000          |
|11     |1        |KN1   |4  |Nadia         |Magelang  |Kopi Nusantara Magelang  |Menu-11  |Kopi          |22000|88000          |
|11     |1        |KN2   |3  |Nadia         |Magelang  |Kopi Nusantara Magelang  |Menu-11  |Kopi          |22000|66000          |
|11     |3        |KN3   |1  |Nadia         |Semarang  |Kopi Nusantara Semarang  |Menu-11  |Kopi          |22000|22000          |
|12     |2        |KN4   |4  |Reza          |Yogyakarta|Kopi Nusantara Yogyakarta|Menu-12 

In [16]:
# C. TRANSFORM — Window Function
ringkasan_menu_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())
top2_menu = ringkasan_menu_cabang.withColumn("peringkat", row_number().over(window_cabang)) \
                                    .filter(col("peringkat") <= 2)

top2_menu.show(truncate=False)

+-------------------------+---------+---------------+---------+
|nama_cabang              |nama_menu|total_penjualan|peringkat|
+-------------------------+---------+---------------+---------+
|Kopi Nusantara Magelang  |Menu-1   |8505000        |1        |
|Kopi Nusantara Magelang  |Menu-12  |8085000        |2        |
|Kopi Nusantara Semarang  |Menu-13  |8750000        |1        |
|Kopi Nusantara Semarang  |Menu-9   |8715000        |2        |
|Kopi Nusantara Yogyakarta|Menu-12  |9275000        |1        |
|Kopi Nusantara Yogyakarta|Menu-15  |8160000        |2        |
+-------------------------+---------+---------------+---------+



In [18]:
# D. TRANSFORM — Spark SQL 
df_gabungan.createOrReplaceTempView("gabungan")

penjualan_kepala_barista = spark.sql('''
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM gabungan
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
''')
penjualan_kepala_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



In [20]:
# E. LOAD
!hdfs dfs -mkdir -p /user/irkham/tugas7/processed

df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(
    "hdfs://localhost:9000/user/irkham/tugas7/processed/laporan"
)
print("Laporan berhasil disimpan ke HDFS dalam format Parquet, terpartisi per kategori menu.")
!hdfs dfs -ls /user/irkham/tugas7/processed/laporan

df_verifikasi = spark.read.parquet("hdfs://localhost:9000/user/irkham/tugas7/processed/laporan")
print("jumlah baris : ", df_verifikasi.count())

[Stage 47:>                                                         (0 + 1) / 1]

Laporan berhasil disimpan ke HDFS dalam format Parquet, terpartisi per kategori menu.


Found 4 items
-rw-r--r--   3 irkham supergroup          0 2026-09-30 21:45 /user/irkham/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - irkham supergroup          0 2026-09-30 21:45 /user/irkham/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - irkham supergroup          0 2026-09-30 21:45 /user/irkham/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - irkham supergroup          0 2026-09-30 21:45 /user/irkham/tugas7/processed/laporan/kategori_menu=Non-Kopi
jumlah baris :  4000


In [25]:
# F. Ringkasan Eksekutif
df_final = spark.read.parquet("hdfs://localhost:9000/user/irkham/tugas7/processed/laporan")

ringkasan_eksekutif = df_final.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi"),    
).orderBy(col("total_penjualan").desc())

ringkasan_eksekutif.show(truncate=False)

+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|nama_cabang              |kota      |kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Semarang  |Semarang  |Nadia         |90542000       |1393            |64997.8463747308         |
|Kopi Nusantara Magelang  |Magelang  |Nadia         |87138000       |1302            |66926.26728110599        |
|Kopi Nusantara Yogyakarta|Yogyakarta|Reza          |84645000       |1305            |64862.06896551724        |
+-------------------------+----------+--------------+---------------+----------------+-------------------------+



Cabang mana yang berkinerja terbaik, menu apa yang sebaiknya dipromosikan lebih gencar di cabang dengan kinerja terlemah (berdasarkan temuan bagian C), dan alasannya.
Jawab:
Berdasarkan hasil pengolahan data transaksi Kopi Nusantara, terdapat tiga cabang yang dianalisis, yaitu Magelang, Yogyakarta, dan Semarang. Cabang Semarang memiliki total penjualan tertinggi sebesar Rp90.542.000 dengan jumlah 1.393 transaksi. Cabang Magelang berada pada posisi berikutnya dengan total penjualan Rp87.138.000 dari 1.302 transaksi, sedangkan Yogyakarta memiliki total penjualan Rp84.645.000 dari 1.305 transaksi. Berdasarkan hasil window function, Menu-13 dan Menu-9 menjadi dua menu dengan penjualan tertinggi di cabang Semarang. Keduanya termasuk kategori Makanan Ringan dengan total penjualan masing-masing Rp8.750.000 dan Rp8.715.000. Oleh karena itu, untuk cabang dengan total penjualan terendah, yaitu Yogyakarta, menu yang dapat dipertimbangkan untuk dipromosikan lebih gencar adalah Menu-12 karena menjadi menu dengan penjualan tertinggi di cabang tersebut, yaitu Rp9.275.000. Selain itu, Menu-15 juga dapat dipertimbangkan karena menempati peringkat kedua dengan penjualan Rp8.160.000. Promosi dapat difokuskan pada menu yang sudah menunjukkan permintaan tinggi sehingga berpotensi menarik lebih banyak transaksi.

In [36]:
spark.stop()
print("SparkSession ditutup")

SparkSession ditutup
